# DistilBERT baseline on real AG News data

Run `src/prepare_data.py` first. This notebook trains on the 8,000 real training articles and selects a checkpoint using the 2,000 article validation split. The original test split is not used during training.

In [1]:
from pathlib import Path

from datasets import load_from_disk
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'ag_news'
MODEL_DIR = PROJECT_ROOT / 'models' / 'distilbert_ag_news_baseline'
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints' / 'distilbert_ag_news_baseline'
SEED = 42
MODEL_NAME = 'distilbert/distilbert-base-uncased'
set_seed(SEED)

splits = load_from_disk(str(DATA_DIR))
label_names = splits['train'].features['label'].names
assert len(splits['train']) == 8_000
assert len(splits['validation']) == 2_000
assert len(splits['test']) == 7_600
assert len(label_names) == 4
print({name: len(split) for name, split in splits.items()})
print('Labels:', label_names)

E:\VGTU\5 semester\Natural Language Processing\lab 4\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'train': 8000, 'validation': 2000, 'test': 7600}
Labels: ['World', 'Sports', 'Business', 'Sci/Tech']


In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label={index: name for index, name in enumerate(label_names)},
    label2id={name: index for index, name in enumerate(label_names)},
)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=128)

tokenized = splits.select_columns(['text', 'label']).map(
    tokenize, batched=True, remove_columns=['text']
)
print(tokenized)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]


Loading weights:  22%|██▏       | 22/100 [00:00<00:00, 199.02it/s]


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 853.43it/s]


[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

...


Map: 100%|██████████| 2000/2000 [00:00<00:00, 22484.80 examples/s]


Map:   0%|          | 0/7600 [00:00<?, ? examples/s]


Map:  53%|█████▎    | 4000/7600 [00:00<00:00, 37506.99 examples/s]


Map: 100%|██████████| 7600/7600 [00:00<00:00, 32763.86 examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8000
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7600
    })
})


In [3]:
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    weight_decay=0.01,
    eval_strategy='steps',
    eval_steps=250,
    save_strategy='steps',
    save_steps=250,
    logging_strategy='steps',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=2,
    seed=SEED,
    data_seed=SEED,
    report_to='none',
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
)
print('Training examples:', len(trainer.train_dataset))
print('Validation examples:', len(trainer.eval_dataset))
print('Checkpoints:', CHECKPOINT_DIR)

Training examples: 8000
Validation examples: 2000
Checkpoints: E:\VGTU\5 semester\Natural Language Processing\lab 4\checkpoints\distilbert_ag_news_baseline


In [4]:
train_result = trainer.train()
print('Training loss:', train_result.training_loss)
print('Best checkpoint:', trainer.state.best_model_checkpoint)
print('Best validation loss:', trainer.state.best_metric)

E:\VGTU\5 semester\Natural Language Processing\lab 4\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss
250,0.383967,0.356695
500,0.254443,0.294678



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.08it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.02it/s]

E:\VGTU\5 semester\Natural Language Processing\lab 4\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.04it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.95it/s]

Training loss: 0.3499181957244873
Best checkpoint: E:\VGTU\5 semester\Natural Language Processing\lab 4\checkpoints\distilbert_ag_news_baseline\checkpoint-500
Best validation loss: 0.2946779131889343


In [5]:
import json

MODEL_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))
configuration = {
    'source_dataset': 'fancyzhx/ag_news',
    'prepared_data': str(DATA_DIR),
    'model_name': MODEL_NAME,
    'seed': SEED,
    'training_samples': len(splits['train']),
    'validation_samples': len(splits['validation']),
    'test_samples_unused': len(splits['test']),
    'label_names': label_names,
    'max_length': 128,
    'num_train_epochs': training_args.num_train_epochs,
    'train_batch_size': training_args.per_device_train_batch_size,
    'eval_batch_size': training_args.per_device_eval_batch_size,
    'learning_rate': training_args.learning_rate,
    'weight_decay': training_args.weight_decay,
    'best_checkpoint': trainer.state.best_model_checkpoint,
    'best_validation_loss': trainer.state.best_metric,
}
(MODEL_DIR / 'training_config.json').write_text(
    json.dumps(configuration, indent=2), encoding='utf-8'
)
(MODEL_DIR / 'training_history.json').write_text(
    json.dumps(trainer.state.log_history, indent=2), encoding='utf-8'
)
print('Saved best model and training configuration to', MODEL_DIR)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.22it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.19it/s]

Saved best model and training configuration to E:\VGTU\5 semester\Natural Language Processing\lab 4\models\distilbert_ag_news_baseline
